# Paper 3 reproducibility notebook

Public, sanitized notebook for the aggregate methodological release `v1.1-paper3`.

This notebook reproduces metrics that can be calculated from public aggregate tables and clearly marks values that are documented only as restricted-workflow aggregates. It contains no personal records, addresses, coordinates, archival images, row-level discrepancies, credentials, or local author paths.


In [ ]:
from csv import DictReader
from pathlib import Path

ROOT = Path.cwd()
DATA = ROOT / 'paper3_metodos' / 'data'
if not DATA.exists():
    DATA = Path('..') / 'data'

def read_csv(name):
    with (DATA / name).open(newline='', encoding='utf-8') as fh:
        return list(DictReader(fh))

def pct(num, den):
    return round((num / den) * 100, 2)

summary = {row['metric']: row for row in read_csv('validation_summary.csv')}
fields = read_csv('concordance_by_field.csv')
years = read_csv('concordance_by_year.csv')
by_field = {row['field']: row for row in fields}
by_year = {int(row['year']): row for row in years}
sensitivity = read_csv('sensitivity_summary.csv')
reviewers = read_csv('reviewer_agreement_summary.csv')
idem = read_csv('idem_resolution_summary.csv')
bootstrap = read_csv('bootstrap_summary.csv')

print(f'Data directory: {DATA.resolve()}')


## Corpus and independent validation sample


In [ ]:
corpus_total = int(summary['corpus_total']['value'])
sample_records = int(summary['independent_sample']['value'])
conceptual_comparisons = int(summary['conceptual_comparisons']['value'])

assert corpus_total == 1438
assert sample_records == 180
assert conceptual_comparisons == 1980

print({'corpus_total': corpus_total, 'sample_records': sample_records, 'conceptual_comparisons': conceptual_comparisons})


## Semantic/resolved agreement from public aggregate tables


In [ ]:
comparisons = sum(int(row['comparisons']) for row in fields)
matches = sum(int(row['matches']) for row in fields)
disagreements = sum(int(row['disagreements']) for row in fields)
agreement = pct(matches, comparisons)

year_comparisons = sum(int(row['comparisons']) for row in years)
year_matches = sum(int(row['matches']) for row in years)
year_disagreements = sum(int(row['disagreements']) for row in years)

assert comparisons == 1980
assert matches == 1867
assert disagreements == 113
assert agreement == 94.29

assert year_comparisons == comparisons
assert year_matches == matches
assert year_disagreements == disagreements

assert int(by_year[1912]['disagreements']) == 59
assert float(by_year[1912]['agreement_percent']) == 82.12
assert int(by_year[1914]['disagreements']) == 24
assert float(by_year[1914]['agreement_percent']) == 92.73

assert int(by_field['padre_tutor']['disagreements']) == 31
assert int(by_field['domicilio']['disagreements']) == 20

print({'matches': matches, 'comparisons': comparisons, 'disagreements': disagreements, 'semantic_resolved_agreement_percent': agreement})


## Annual composition-weighted agreement


In [ ]:
year_corpus = sum(int(row['corpus_records']) for row in years)
year_sample = sum(int(row['sample_records']) for row in years)
weighted = sum(
    (int(row['matches']) / int(row['comparisons'])) * (int(row['corpus_records']) / year_corpus)
    for row in years
)
weighted_percent = round(weighted * 100, 2)

assert year_corpus == corpus_total
assert year_sample == sample_records
assert weighted_percent == 94.41
assert float(summary['annual_composition_weighted_agreement']['value']) == 94.41

print({'annual_composition_weighted_agreement_percent': weighted_percent})


## Sensitivity excluding empty-empty comparisons


In [ ]:
sens = {row['scenario']: row for row in sensitivity}
non_empty = sens['excluding_empty_empty']
excluded_empty_empty = int(non_empty['excluded_comparisons'])
non_empty_comparisons = int(non_empty['comparisons'])
non_empty_matches = int(non_empty['matches'])
non_empty_agreement = pct(non_empty_matches, non_empty_comparisons)

assert excluded_empty_empty == 295
assert non_empty_comparisons == 1685
assert non_empty_matches == 1572
assert non_empty_agreement == 93.29

print({'excluded_empty_empty': excluded_empty_empty, 'sensitivity_percent': non_empty_agreement})


## Double review and contextual repetition/IDEM checks


In [ ]:
review = reviewers[0]
review_records = int(review['records'])
reviewer_comparisons = int(review['comparisons'])
reviewer_matches = int(review['exact_matches'])
reviewer_agreement = pct(reviewer_matches, reviewer_comparisons)

idem_row = idem[0]
idem_marks = int(idem_row['marks_evaluated'])
idem_correct = int(idem_row['resolved_correctly'])
idem_unresolved_or_incorrect = int(idem_row['unresolved_or_incorrect'])
idem_success = pct(idem_correct, idem_marks)

assert review_records == 60
assert reviewer_comparisons == 1080
assert reviewer_matches == 1080
assert reviewer_agreement == 100.00

assert idem_marks == 398
assert idem_correct == 397
assert idem_unresolved_or_incorrect == 1
assert idem_success == 99.75

print({'double_review_agreement_percent': reviewer_agreement, 'idem_resolution_success_percent': idem_success})


## Bootstrap interval

The bootstrap interval is released as an aggregate from the restricted validation workflow. It belongs to the annual-composition-weighted agreement estimate, not to the unweighted semantic/resolved agreement. The restricted workflow used 5,000 stratified replicates with seed `20260827`.

The public notebook does not recompute bootstrap replicates from scratch. That is not possible from the public aggregate tables because row-level resampling inputs are restricted and are not published.


In [ ]:
boot = bootstrap[0]
bootstrap_estimate = float(boot['estimate_percent'])
bootstrap_lower = float(boot['ci95_lower_percent'])
bootstrap_upper = float(boot['ci95_upper_percent'])
bootstrap_replicates = int(boot['replicates'])

assert boot['measure'] == 'annual_composition_weighted_agreement'
assert bootstrap_estimate == 94.41
assert bootstrap_lower == 92.70
assert bootstrap_upper == 95.98
assert bootstrap_replicates == 5000

print({'bootstrap_estimate_percent': bootstrap_estimate, 'bootstrap_ci95_percent': (bootstrap_lower, bootstrap_upper), 'replicates': bootstrap_replicates})


## Interpretation note

These metrics are not CER, WER, or HTR engine accuracy. The evaluated object is the full pipeline:

transcription structured -> contextual resolution -> normalization -> QA -> human validation.

A correctly expanded `IDEM` mark counts as semantic concordance, not as an error, under the original transcription contract.
